# GPT-2 Training on Google Colab## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted!")

## Install Dependencies

In [ ]:
!pip install torch tiktoken datasets -q
print("Dependencies installed!")

## Import & Training

In [ ]:
import torch
import main

main.setup_environment("cuda")

from mygpt_2 import GPT, GPTConfig, SimpleTokenizer
from mygpt_2.train import create_dataset, train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
is_gpu = device.type == "cuda"

config = GPTConfig(
    d_model=256 if is_gpu else 128,
    num_heads=8 if is_gpu else 4,
    num_layers=8 if is_gpu else 4,
    max_seq_len=512 if is_gpu else 256,
    batch_size=4 if is_gpu else 2,
    grad_accum_steps=4 if is_gpu else 2,
    max_steps=5000,
    warmup_steps=200,
)

tokenizer = SimpleTokenizer()
texts = main.load_data()
dataset = create_dataset(texts[:500], tokenizer, config.max_seq_len)
print(f"Dataset: {len(dataset):,} batches")

model = GPT(config)
print(f"Model parameters: {model.get_num_params():,}")

model = train(model, dataset, config, device, save_dir="/content/checkpoints")
torch.save(model.state_dict(), "/content/gpt_model.pt")
print("Done!")

## Download Checkpoints

In [ ]:
!zip -r /content/drive/My\ Drive/checkpoints.zip /content/checkpoints/
print("Saved to Drive!")